In [13]:
!apt-get update && apt-get install aria2 ffmpeg -y
!pip install internetarchive

zsh:1: command not found: apt-get
Defaulting to user installation because normal site-packages is not writeable


In [20]:
import os
import subprocess
import json
import shutil 
import re
import threading
import queue
import sys
#!{sys.executable} -m pip install internetarchive
#!ffmpeg -version
#!aria2c -v
import internetarchive as ia

# ---------------- CONFIG ----------------
print(os.getcwd())
BASE_DIR = "./EpisodeConversion"
os.chdir(BASE_DIR)

ARCHIVE_ID = "cc-dragon-ball-archive"
TORRENT_FILE = "SoM_Dragon_Ball_DBOX_CC.torrent"

TEMP_DOWNLOADS = os.path.join(BASE_DIR, "temp_mkv")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
JSON_LOG_PATH = os.path.join(BASE_DIR, "episodes", "db.json")

os.environ['IAS3_ACCESS_KEY'] = "GGSYh3cuQXjfZrCP"
os.environ['IAS3_SECRET_KEY'] = "MP15GeRtfhbclDBd"

os.makedirs(os.path.dirname(JSON_LOG_PATH), exist_ok=True)

for d in [TEMP_DOWNLOADS, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# ---------------- QUEUES ----------------

convert_q = queue.Queue()
upload_q = queue.Queue()

# ---------------- UTILS ----------------

def check_disk_space():
    total, used, free = shutil.disk_usage(".")
    free_gb = free // (2**30)
    print(f"Free Disk Space: {free_gb}GB")
    return free_gb > 10


def valid_video(path):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_streams", path],
        capture_output=True,
        text=True
    )
    return result.returncode == 0


def get_audio_streams(file_path):
    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "a",
        "-show_entries", "stream=index:stream_tags=language",
        "-of", "json",
        file_path
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    try:
        data = json.loads(result.stdout)
    except:
        return []

    streams = []
    for i, stream in enumerate(data.get("streams", [])):
        lang = stream.get("tags", {}).get("language", "und")
        streams.append((i, lang))

    return streams


# ---------------- TORRENT PARSER ----------------

def get_sorted_torrent_files():
    print("--- Parsing Torrent Metadata ---")

    result = subprocess.run(
        ["aria2c", "-S", TORRENT_FILE],
        capture_output=True,
        text=True
    )

    files = []

    # matches: "  2|./[SoM] Dragon Ball DBOX CC/Dragon.Ball.002.DBOX.CC.480p.x264-SoM.mkv"
    pattern = re.compile(r'^\s*(\d+)\|(.+?\.mkv)', re.MULTILINE)
    ep_pattern = re.compile(r'Dragon\.Ball\.(\d{1,3})\.DBOX', re.IGNORECASE)

    for match in pattern.finditer(result.stdout):
        idx = int(match.group(1))
        path = match.group(2).strip()
        filename = os.path.basename(path)

        ep = ep_pattern.search(filename)
        if ep:
            files.append({
                "index": idx,
                "name": filename,
                "episode": int(ep.group(1))
            })

    files.sort(key=lambda x: x["episode"])

    print(f"Found {len(files)} episodes")
    return files


# ---------------- DOWNLOAD ----------------

def download_from_torrent(file_index, episode):
    temp_dir = os.path.join(TEMP_DOWNLOADS, f"ep_{episode}")
    shutil.rmtree(temp_dir, ignore_errors=True)
    os.makedirs(temp_dir, exist_ok=True)

    cmd = [
        "aria2c",
        f"--select-file={file_index}",
        "--seed-time=0",
        "--max-connection-per-server=16",
        "--split=16",
        "--min-split-size=10M",
        "-d", temp_dir,
        TORRENT_FILE
    ]

    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL)

    for root, dirs, files in os.walk(temp_dir):
        for f in files:
            if f.endswith(".mkv"):
                return os.path.join(root, f), temp_dir

    return None, temp_dir


# ---------------- HLS CONVERSION ----------------

def convert_to_hls(input_path, output_folder):
    print(f"[CONVERT] {os.path.basename(input_path)}")

    os.makedirs(output_folder, exist_ok=True)

    # get all audio streams
    streams = get_audio_streams(input_path)

    cmd = [
        "ffmpeg",
        "-y",
        "-fflags", "+genpts",
        "-i", input_path,
        "-map", "0:v:0",
    ]

    # map all audio tracks (optional each, so missing ones don't crash)
    for stream_index, _lang in streams:
        cmd += ["-map", f"0:a:{stream_index}?"]

    cmd += [
        "-sn",
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "160k",
        "-f", "hls",
        "-hls_time", "20",
        "-hls_playlist_type", "vod",
        "-hls_segment_filename",
        os.path.join(output_folder, "segment_%03d.ts"),
        os.path.join(output_folder, "master.m3u8")
    ]

    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL)


# ---------------- MANIFEST ----------------

def load_manifest():
    if not os.path.exists(JSON_LOG_PATH):
        with open(JSON_LOG_PATH, "w") as f:
            json.dump([], f)

    with open(JSON_LOG_PATH) as f:
        return json.load(f)


def save_manifest(manifest):
    with open(JSON_LOG_PATH, "w") as f:
        json.dump(manifest, f, indent=4)


# ---------------- THREADS ----------------

def downloader(files):
    for item in files:
        if not check_disk_space():
            break

        mkv, temp_dir = download_from_torrent(item["index"], item["episode"])

        if mkv and valid_video(mkv):
            convert_q.put((mkv, item, temp_dir))
        else:
            shutil.rmtree(temp_dir, ignore_errors=True)

    convert_q.put(None)



def converter(series_name):
    root = os.path.join(OUTPUT_DIR, series_name)
    os.makedirs(root, exist_ok=True)

    while True:
        item = convert_q.get()

        if item is None:
            upload_q.put(None)
            break

        mkv, meta, temp_dir = item
        filename = meta["name"]
        slug = (
            filename
            .replace(".mkv", "")
            .replace(".", "_")
            .replace(" ", "_")
        )
        
        out = os.path.join(root, slug)

        try:
            convert_to_hls(mkv, out)
            upload_q.put((out, meta, slug))
        except Exception as e:
            print("CONVERT ERROR:", e)

        shutil.rmtree(temp_dir, ignore_errors=True)


def uploader():
    manifest = load_manifest()

    while True:
        item = upload_q.get()

        if item is None:
            break

        path, meta, slug = item
        filename = meta["name"]

        print(f"[UPLOAD] {slug}")

        files = {}
        for f in os.listdir(path):
            files[f"{slug}/{f}"] = os.path.join(path, f)

        try:
            ia.upload(
                ARCHIVE_ID,
                files,
                metadata={"mediatype": "movies"},
                access_key=ACCESS,
                secret_key=SECRET,
                verbose=False
            )

            url = f"https://archive.org/download/{ARCHIVE_ID}/{slug}/master.m3u8"

            if not any(e["id"] == meta["episode"] for e in manifest):
                manifest.append({
                    "id": meta["episode"],
                    "title": filename.replace(".mkv", "").replace(".", " "),
                    "hls": url
                })

            save_manifest(manifest)

            print("[UPLOADER SUCCESSFUL]", slug)

        except Exception as e:
            print("UPLOAD ERROR:", e)

        shutil.rmtree(path, ignore_errors=True)


# ---------------- PIPELINE ----------------

def run_pipeline(series, start, count):
    files = get_sorted_torrent_files()
    target = files[start - 1:start - 1 + count]

    t1 = threading.Thread(target=downloader, args=(target,))
    t2 = threading.Thread(target=converter, args=(series,))
    t3 = threading.Thread(target=uploader)

    t1.start()
    t2.start()
    t3.start()

    t1.join()
    t2.join()
    t3.join()

    print("Pipeline finished")


# ---------------- RUN ----------------

if __name__ == "__main__":
    run_pipeline("Dragon_Ball", 1, 2)


/


FileNotFoundError: [Errno 2] No such file or directory: './EpisodeConversion'